# Libs

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

from mealpy import FloatVar, IntegerVar, ES, PSO

# Load Dataset

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.reshape(-1, 784).astype("float32") / 255.0
x_test = x_test.reshape(-1, 784).astype("float32") / 255.0

print("Training data shape:", x_train.shape)
print("Test data shape:", x_test.shape)

# Set functions

In [ ]:
def build_autoencoder_classifier(params):
    n1 = int(round(params[0]))  # camada 1
    n2 = int(round(params[1]))  # camada 2
    n3 = int(round(params[2]))  # camada 3
    lr = 0.0001      # learning rate

    input_layer = Input(shape=(784,))

    # Encoder
    x = Dense(n1, activation='relu')(input_layer)
    x = Dense(n2, activation='relu')(x)
    x = Dense(n3, activation='relu')(x)
    latent = Dense(32, activation='relu')(x)

    # Decoder
    x = Dense(n3, activation='relu')(latent)
    x = Dense(n2, activation='relu')(x)
    x = Dense(n1, activation='relu')(x)
    output_layer = Dense(784, activation='sigmoid')(x)

    autoencoder = Model(input_layer, output_layer)
    encoder = Model(input_layer, latent)
    autoencoder.compile(optimizer=Adam(lr), loss='mse')

    return autoencoder, encoder

def build_classifier(encoder):
    for layer in encoder.layers:
        layer.trainable = False
    input_latent = encoder.input
    x = encoder.output
    x = Dense(64, activation='relu')(x)
    output = Dense(10, activation='softmax')(x)

    model = Model(input_latent, output)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def objective_function(solution):
    try:
        # garantir que os parâmetros estão dentro dos limites
        n1 = int(np.clip(solution[0], 200, 400))
        n2 = int(np.clip(solution[1], 100, 200))
        n3 = int(np.clip(solution[2], 20, 100))
        lr = 0.0001

        autoencoder, encoder = build_autoencoder_classifier([n1, n2, n3, lr])
        autoencoder.fit(x_train, x_train, epochs=10, batch_size=256, verbose=0)

        classifier = build_classifier(encoder)
        classifier.fit(x_train, y_train, epochs=10, batch_size=256, verbose=0)

        preds = classifier.predict(x_test)
        acc = accuracy_score(y_test, np.argmax(preds, axis=1))

        return 1 - acc  # minimizar
    except Exception as e:
        print("Erro ao avaliar indivíduo:", solution)
        print("Detalhes:", str(e))
        return 1.0

# Set params for the problem

In [ ]:
problem_dict = {
    "bounds": [
        IntegerVar(lb=200, ub=400, name="neurons_layer1"),
        IntegerVar(lb=100, ub=200, name="neurons_layer2"),
        IntegerVar(lb=20, ub=100, name="neurons_layer3")
    ],
    "minmax": "min",
    "obj_func": objective_function
}

# PSO

## Run PSO

In [ ]:
model = PSO.OriginalPSO(epoch=25, pop_size=15, c1=2.05, c2=2.5, w=0.4, verbose=True)
g_best = model.solve(problem_dict)

## PSO Results

In [ ]:
# Extrair valores da melhor solução
neurons_layer1 = int(round(g_best.solution[0]))
neurons_layer2 = int(round(g_best.solution[1]))
neurons_layer3 = int(round(g_best.solution[2]))
learning_rate = 0.0001

print("\nMelhor solução encontrada:")
print(f"Neurônios camada 1: {neurons_layer1}")
print(f"Neurônios camada 2: {neurons_layer2}")
print(f"Neurônios camada 3: {neurons_layer3}")
print(f"Taxa de aprendizado: {learning_rate:.4f}")
print(f"Fitness (1 - acurácia): {g_best.target.fitness:.5f}")

# Plotando curva de convergência
plt.plot(model.history.list_global_best_fit)
plt.title("Convergência do PSO (MEALpy - OriginalPSO)")
plt.xlabel("Geração")
plt.ylabel("Fitness (1 - Acurácia)")
plt.grid(True)
plt.show()


## PSO Metrics

In [ ]:
# Recuperando os melhores parâmetros encontrados pelo PSO
best_n1 = int(np.clip(g_best.solution[0], 200, 400))
best_n2 = int(np.clip(g_best.solution[1], 100, 200))
best_n3 = int(np.clip(g_best.solution[2], 20, 100))
best_lr = 0.0001   # fixo

print("\n--- Avaliação final com o melhor modelo ---")
print("Camadas ocultas: 3 (fixo)")
print(f"Neurônios por camada: [{best_n1}, {best_n2}, {best_n3}]")
print(f"Learning rate: {best_lr:.4f}")

# Reconstruir e treinar com os melhores hiperparâmetros
autoencoder, encoder = build_autoencoder_classifier([best_n1, best_n2, best_n3, best_lr])
autoencoder.fit(x_train, x_train, epochs=25, batch_size=256, verbose=0)

classifier = build_classifier(encoder)
classifier.fit(x_train, y_train, epochs=25, batch_size=256, verbose=0)

# Previsões e métricas
y_pred = classifier.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis=1)

acc = accuracy_score(y_test, y_pred_classes)
prec = precision_score(y_test, y_pred_classes, average='macro')
rec = recall_score(y_test, y_pred_classes, average='macro')
f1 = f1_score(y_test, y_pred_classes, average='macro')

print(f"\nAcurácia final:  {acc:.4f}")
print(f"Precisão:        {prec:.4f}")
print(f"Recall:       {rec:.4f}")
print(f"F1-Score:        {f1:.4f}")

## Plot reconstructed images

In [ ]:
n = 10  # número de imagens
imgs = x_test[:n]

# Reconstruções feitas pelo autoencoder
reconstructed = autoencoder.predict(imgs)

plt.figure(figsize=(20, 4))
for i in range(n):
    # Imagem original
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(imgs[i].reshape(28, 28), cmap="gray")
    plt.title("Original")
    plt.axis("off")
    
    # Imagem reconstruída
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(reconstructed[i].reshape(28, 28), cmap="gray")
    plt.title("Reconstruída")
    plt.axis("off")

plt.show()